# Lesson 12B: Variational Autoencoder - Practical

**VAE** learns probabilistic latent space for generation.

**Loss** = Reconstruction + KL Divergence

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

torch.manual_seed(42)
print('✅ Loaded!')

In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 32)
        self.fc_mu = nn.Linear(32, 8)
        self.fc_logvar = nn.Linear(32, 8)
        self.fc2 = nn.Linear(8, 32)
        self.fc3 = nn.Linear(32, 64)
    
    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)
    
    def decode(self, z):
        return torch.sigmoid(self.fc3(F.relu(self.fc2(z))))
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

vae = VAE()
print('✅ VAE defined')

In [ ]:
# Train VAE
digits = load_digits()
X = torch.FloatTensor(digits.data / 16.0)
optimizer = torch.optim.Adam(vae.parameters(), lr=0.001)

for epoch in range(50):
    recon, mu, logvar = vae(X)
    recon_loss = F.mse_loss(recon, X, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    loss = recon_loss + kld
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}, Loss: {loss.item():.0f}')
print('✅ VAE trained!')